# BioRAG-X — 06 Dense Embeddings & ANN

## MedCPT + Exact Dense Retrieval + HNSW + IVF + Late-Chunking Embeddings

This notebook connects our chunking work to **real dense retrieval**.

### Objectives
- Understand biomedical embeddings.
- Use MedCPT query/article encoders.
- Build an exact dense-search reference.
- Build ANN indexes with HNSW and IVF.
- Measure recall/latency trade-offs.
- Define the actual late-chunking embedding experiment.
- Persist reusable vector artifacts.

### Core research question
> Can a biomedical dense retriever recover BioASQ evidence reliably, and can ANN reduce retrieval cost while preserving most of the exact-search recall?

**Rule:** exact dense search is the ANN quality reference.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib, json, math, time, re
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

CANONICAL_DIR = Path("data/canonical")
ARTIFACT_DIR = Path("artifacts/06_dense_embeddings_and_ann")
INDEX_DIR = Path("data/indexes/dense")

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

PASSAGES_PATH = CANONICAL_DIR / "passages.parquet"
QUESTIONS_PATH = CANONICAL_DIR / "questions.parquet"
GOLD_REL_PATH = CANONICAL_DIR / "gold_relationships.parquet"

if not PASSAGES_PATH.exists():
    raise FileNotFoundError("Run Notebook 02 first.")

passages = pd.read_parquet(PASSAGES_PATH)
questions = pd.read_parquet(QUESTIONS_PATH)
gold_relationships = pd.read_parquet(GOLD_REL_PATH)

print("Passages:", len(passages))
print("Questions:", len(questions))
print("Gold relationships:", len(gold_relationships))

## 1. Exact dense retrieval is our control condition

Given normalized vectors:

`cosine(q,d) = q · d`

Exact retrieval compares the query with every passage vector and returns the true top-K.

ANN will be judged against this result.

Without an exact reference, we cannot honestly say how much recall an ANN index loses.

In [ ]:
def normalize_embeddings(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.clip(np.linalg.norm(x, axis=1, keepdims=True), 1e-12, None)

def exact_topk(query_vector, matrix, k=10):
    q = np.asarray(query_vector, dtype=np.float32)
    q = q / max(np.linalg.norm(q), 1e-12)
    scores = matrix @ q
    k = min(k, len(scores))
    idx = np.argpartition(-scores, k-1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return idx, scores[idx]

## 2. MedCPT

Configured biomedical encoders:

- Query: `ncbi/MedCPT-Query-Encoder`
- Passage/article: `ncbi/MedCPT-Article-Encoder`

We use separate query and document encoders as intended by MedCPT.

For every vector artifact we record:
- model name
- backend
- pooling
- normalization
- dimension
- source snapshot

In [ ]:
MEDCPT_QUERY_MODEL = "ncbi/MedCPT-Query-Encoder"
MEDCPT_ARTICLE_MODEL = "ncbi/MedCPT-Article-Encoder"

try:
    import torch
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_OK = True
except ImportError:
    TRANSFORMERS_OK = False

print("Transformers available:", TRANSFORMERS_OK)

In [ ]:
class MedCPTEmbedder:
    def __init__(self):
        if not TRANSFORMERS_OK:
            raise RuntimeError("Install torch + transformers for real MedCPT embeddings.")
        self.qt = AutoTokenizer.from_pretrained(MEDCPT_QUERY_MODEL)
        self.qm = AutoModel.from_pretrained(MEDCPT_QUERY_MODEL)
        self.dt = AutoTokenizer.from_pretrained(MEDCPT_ARTICLE_MODEL)
        self.dm = AutoModel.from_pretrained(MEDCPT_ARTICLE_MODEL)
        self.qm.eval(); self.dm.eval()

    @staticmethod
    def mean_pool(hidden, mask):
        m = mask.unsqueeze(-1).float()
        return (hidden*m).sum(1) / m.sum(1).clamp(min=1e-9)

    def _encode(self, texts, tokenizer, model, max_length, batch_size):
        device = next(model.parameters()).device
        out = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(
                batch, padding=True, truncation=True,
                max_length=max_length, return_tensors="pt"
            ).to(device)
            with torch.no_grad():
                h = model(**inputs).last_hidden_state
                pooled = self.mean_pool(h, inputs["attention_mask"])
            out.append(pooled.detach().cpu().numpy())
        return normalize_embeddings(np.vstack(out))

    def encode_queries(self, texts, max_length=256, batch_size=16):
        return self._encode(texts, self.qt, self.qm, max_length, batch_size)

    def encode_documents(self, texts, max_length=512, batch_size=16):
        return self._encode(texts, self.dt, self.dm, max_length, batch_size)

## 3. Development backend

For notebook testing, a deterministic mock backend is supported.

**Mock vectors are never reported as MedCPT scientific results.**

They only validate:
- batching
- storage
- search
- ANN wiring
- evaluation code

In [ ]:
def mock_embedding(text, dim=128):
    seed = int(hashlib.sha256(text.encode()).hexdigest()[:16], 16) % (2**32)
    rng = np.random.default_rng(seed)
    v = rng.normal(size=dim).astype(np.float32)
    return v / max(np.linalg.norm(v), 1e-12)

def mock_encode(texts, dim=128):
    return np.vstack([mock_embedding(t, dim) for t in texts])

## 4. Development corpus

Start with a deterministic sample. The final benchmark must use a frozen corpus snapshot.

All downstream dense experiments must use the **same passages and same embedding artifact** so comparisons remain controlled.

In [ ]:
DENSE_SAMPLE_N = min(10000, len(passages))
dense_passages = passages.sample(n=DENSE_SAMPLE_N, random_state=SEED).reset_index(drop=True)

available = set(dense_passages["canonical_passage_id"])
dev_questions = questions[
    questions["gold_canonical_passage_ids"].map(lambda ids: all(pid in available for pid in ids))
].copy().reset_index(drop=True)

print("Development passages:", len(dense_passages))
print("Questions with complete gold evidence in sample:", len(dev_questions))

In [ ]:
use_real_medcpt = False  # Set True after model download/access is validated.

if use_real_medcpt:
    embedder = MedCPTEmbedder()
    passage_embeddings = embedder.encode_documents(
        dense_passages["retrieval_text"].tolist(),
        max_length=512,
        batch_size=16
    )
    embedding_backend = "medcpt"
else:
    passage_embeddings = mock_encode(
        dense_passages["retrieval_text"].tolist(), dim=128
    )
    embedding_backend = "deterministic_mock"

print("Backend:", embedding_backend)
print("Embedding matrix:", passage_embeddings.shape)

In [ ]:
if embedding_backend == "medcpt":
    query_embeddings = embedder.encode_queries(
        dev_questions["question"].tolist(),
        max_length=256,
        batch_size=16
    )
else:
    query_embeddings = mock_encode(
        dev_questions["question"].tolist(),
        dim=passage_embeddings.shape[1]
    )

passage_ids = dense_passages["canonical_passage_id"].tolist()

embedding_manifest = {
    "backend": embedding_backend,
    "query_encoder": MEDCPT_QUERY_MODEL,
    "document_encoder": MEDCPT_ARTICLE_MODEL,
    "dimension": int(passage_embeddings.shape[1]),
    "normalized": True,
    "passage_count": len(dense_passages),
    "query_count": len(dev_questions),
}
(ARTIFACT_DIR / "embedding_manifest.json").write_text(
    json.dumps(embedding_manifest, indent=2),
    encoding="utf-8"
)

np.save(INDEX_DIR / f"passage_embeddings_{embedding_backend}.npy", passage_embeddings)
dense_passages[["canonical_passage_id"]].to_parquet(
    INDEX_DIR / f"passage_ids_{embedding_backend}.parquet",
    index=False
)

## 5. Retrieval metrics

Because the dataset contains gold passage IDs, compute:

- Hit@K
- Recall@K
- MRR

For biomedical multi-passage questions, **Recall@K** is particularly important.

In [ ]:
def retrieval_metrics(retrieved_ids, gold_ids, k_values=(1,5,10,20)):
    gold = set(gold_ids)
    out = {}
    for k in k_values:
        top = retrieved_ids[:k]
        hits = len(set(top) & gold)
        out[f"hit@{k}"] = float(hits > 0)
        out[f"recall@{k}"] = hits / max(1, len(gold))

    rr = 0.0
    for rank, pid in enumerate(retrieved_ids, 1):
        if pid in gold:
            rr = 1.0 / rank
            break
    out["mrr"] = rr
    return out

## 6. Exact dense retrieval

This establishes the best achievable dense-search result for the current embedding representation.

In [ ]:
def exact_search(query_vectors, passage_vectors, passage_ids, k=20):
    results = []
    for q in query_vectors:
        idx, scores = exact_topk(q, passage_vectors, k)
        results.append([(passage_ids[i], float(s)) for i, s in zip(idx, scores)])
    return results

start = time.perf_counter()
exact_results = exact_search(query_embeddings, passage_embeddings, passage_ids, 20)
exact_ms = 1000 * (time.perf_counter() - start)

rows = []
for qrow, result in zip(dev_questions.itertuples(index=False), exact_results):
    m = retrieval_metrics(
        [pid for pid, _ in result],
        qrow.gold_canonical_passage_ids
    )
    rows.append(m)

exact_eval = pd.DataFrame(rows)
print("Exact batch latency (ms):", exact_ms)
display(exact_eval.mean(numeric_only=True).to_frame("exact_mean"))

# 7. HNSW

HNSW is the primary ANN candidate.

Parameters:
- `M`
- `efConstruction`
- `efSearch`

We vary `efSearch` because it directly controls the recall/latency trade-off.

In [ ]:
try:
    import faiss
    FAISS_OK = True
    print("FAISS:", faiss.__version__)
except ImportError:
    FAISS_OK = False
    print("FAISS unavailable; install faiss-cpu for ANN experiments.")

In [ ]:
def build_hnsw(vectors, M=32, ef_construction=200):
    if not FAISS_OK:
        raise RuntimeError("FAISS required.")
    index = faiss.IndexHNSWFlat(vectors.shape[1], M, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = ef_construction
    index.add(vectors.astype(np.float32))
    return index

def hnsw_search(index, queries, ids, k=20, ef_search=64):
    index.hnsw.efSearch = ef_search
    scores, idx = index.search(queries.astype(np.float32), k)
    return [
        [(ids[j], float(s)) for j, s in zip(row_i, row_s) if j >= 0]
        for row_i, row_s in zip(idx, scores)
    ]

In [ ]:
hnsw_rows = []

if FAISS_OK:
    hnsw = build_hnsw(passage_embeddings, M=32, ef_construction=200)

    for ef in [16, 32, 64, 128]:
        start = time.perf_counter()
        results = hnsw_search(hnsw, query_embeddings, passage_ids, 20, ef)
        elapsed = 1000 * (time.perf_counter() - start)

        per_q = []
        for qrow, result in zip(dev_questions.itertuples(index=False), results):
            per_q.append(retrieval_metrics(
                [pid for pid, _ in result],
                qrow.gold_canonical_passage_ids
            ))

        m = pd.DataFrame(per_q).mean(numeric_only=True).to_dict()
        m.update({
            "index":"HNSW",
            "ef_search":ef,
            "batch_latency_ms":elapsed,
            "avg_latency_ms":elapsed/max(1,len(dev_questions))
        })
        hnsw_rows.append(m)

hnsw_results = pd.DataFrame(hnsw_rows)
display(hnsw_results)

# 8. IVF

IVF partitions the vector space into coarse clusters.

Parameters:
- `nlist`: number of coarse clusters
- `nprobe`: number of clusters searched

Like HNSW:
higher search breadth usually increases recall at higher compute cost.

In [ ]:
def build_ivf(vectors, nlist):
    if not FAISS_OK:
        raise RuntimeError("FAISS required.")
    quantizer = faiss.IndexFlatIP(vectors.shape[1])
    index = faiss.IndexIVFFlat(
        quantizer, vectors.shape[1], nlist, faiss.METRIC_INNER_PRODUCT
    )
    index.train(vectors.astype(np.float32))
    index.add(vectors.astype(np.float32))
    return index

def ivf_search(index, queries, ids, k=20, nprobe=8):
    index.nprobe = nprobe
    scores, idx = index.search(queries.astype(np.float32), k)
    return [
        [(ids[j], float(s)) for j, s in zip(row_i, row_s) if j >= 0]
        for row_i, row_s in zip(idx, scores)
    ]

In [ ]:
ivf_rows = []

if FAISS_OK:
    nlist = min(64, max(4, int(np.sqrt(len(passage_embeddings)))))
    ivf = build_ivf(passage_embeddings, nlist)

    probes = sorted(set([1, 4, 8, 16, min(nlist, 32)]))
    for nprobe in probes:
        start = time.perf_counter()
        results = ivf_search(ivf, query_embeddings, passage_ids, 20, nprobe)
        elapsed = 1000 * (time.perf_counter() - start)

        per_q = []
        for qrow, result in zip(dev_questions.itertuples(index=False), results):
            per_q.append(retrieval_metrics(
                [pid for pid, _ in result],
                qrow.gold_canonical_passage_ids
            ))

        m = pd.DataFrame(per_q).mean(numeric_only=True).to_dict()
        m.update({
            "index":"IVF",
            "nlist":nlist,
            "nprobe":nprobe,
            "batch_latency_ms":elapsed,
            "avg_latency_ms":elapsed/max(1,len(dev_questions))
        })
        ivf_rows.append(m)

ivf_results = pd.DataFrame(ivf_rows)
display(ivf_results)

# 9. ANN recall loss and speedup

Definitions:

```text
ANN recall loss@K
= Exact Recall@K - ANN Recall@K

ANN speedup
= Exact latency / ANN latency
```

We want a useful Pareto point, not blindly the lowest latency.

In [ ]:
exact_r10 = float(exact_eval["recall@10"].mean())

if not hnsw_results.empty:
    h = hnsw_results.copy()
    h["recall_loss@10"] = exact_r10 - h["recall@10"]
    h["speedup_vs_exact"] = exact_ms / h["batch_latency_ms"].replace(0, np.nan)
    display(h[["ef_search","recall@10","recall_loss@10","avg_latency_ms","speedup_vs_exact"]])

if not ivf_results.empty:
    v = ivf_results.copy()
    v["recall_loss@10"] = exact_r10 - v["recall@10"]
    v["speedup_vs_exact"] = exact_ms / v["batch_latency_ms"].replace(0, np.nan)
    display(v[["nlist","nprobe","recall@10","recall_loss@10","avg_latency_ms","speedup_vs_exact"]])

# 10. Actual late-chunking embedding experiment

Late chunking is **not** ordinary chunk embedding.

### Ordinary
`chunk → encode chunk independently`

### Late
`long context → contextual token embeddings → pool token spans → chunk embeddings`

We therefore create the span interface here and defer the actual long-context model run until the environment/model is validated.

A future implementation must record:
- model
- context length
- tokenizer
- span alignment
- pooling
- truncation
- normalization

In [ ]:
LATE_MODEL_CANDIDATE = "jinaai/jina-embeddings-v2-base-en"

def late_chunk_spans(text, chunk_size=256, overlap=32):
    t = tokens(text)
    step = chunk_size - overlap
    out, start, idx = [], 0, 0
    while start < len(t):
        end = min(start + chunk_size, len(t))
        out.append({
            "chunk_index":idx,
            "start_token":start,
            "end_token":end,
            "text":" ".join(t[start:end]),
            "approximate_tokens":end-start
        })
        idx += 1
        if end == len(t): break
        start += step
    return out

print(late_chunk_spans(
    dense_passages.iloc[0]["retrieval_text"], 256, 32
)[:2])

In [ ]:
late_manifest = {
    "model_candidate": LATE_MODEL_CANDIDATE,
    "status": "span_contract_only",
    "reason": "true late chunking requires one long-context encoder pass followed by token-span pooling"
}
(ARTIFACT_DIR / "late_chunking_manifest.json").write_text(
    json.dumps(late_manifest, indent=2), encoding="utf-8"
)

# 11. Persistence and experiment manifest

The dense layer must be reusable by Notebook 07.

Saved artifacts include:
- passage vectors
- passage-ID mapping
- exact evaluation
- HNSW evaluation
- IVF evaluation
- model/backend manifest
- late-chunking contract

No later notebook should silently recompute embeddings with another model.

In [ ]:
exact_eval.to_parquet(ARTIFACT_DIR / "exact_dense_eval.parquet", index=False)
if not hnsw_results.empty:
    hnsw_results.to_parquet(ARTIFACT_DIR / "hnsw_results.parquet", index=False)
if not ivf_results.empty:
    ivf_results.to_parquet(ARTIFACT_DIR / "ivf_results.parquet", index=False)

manifest = {
    "notebook":"06_dense_embeddings_and_ann",
    "seed":SEED,
    "embedding_backend":embedding_backend,
    "query_encoder":MEDCPT_QUERY_MODEL,
    "document_encoder":MEDCPT_ARTICLE_MODEL,
    "embedding_dim":int(passage_embeddings.shape[1]),
    "passage_count":len(dense_passages),
    "query_count":len(dev_questions),
    "ann":["HNSW","IVF"],
    "late_chunking":"contract_defined; validated model run belongs here after long-context backend is confirmed",
    "rules":[
        "Exact dense search is the ANN reference.",
        "Mock vectors are not scientific MedCPT results.",
        "Every result joins through canonical_passage_id."
    ]
}

(ARTIFACT_DIR / "dense_ann_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print("Saved artifacts to", ARTIFACT_DIR)

# 12. Key takeaways

- **Dense retrieval** provides semantic matching.
- **MedCPT** gives a biomedical-specific representation.
- **Exact search** is our quality control condition.
- **HNSW** is the primary production ANN candidate.
- **IVF** is a second ANN/scaling path.
- **Late chunking** is an embedding procedure, not a simple splitter.
- ANN is accepted only if its recall/latency/memory trade-off is justified.

## Handoff to Notebook 07

Next we combine this semantic branch with:

- BM25
- Hybrid retrieval
- RRF
- Candidate-pool fusion
- Cross-encoder reranking

and begin answering the more important question:

> **Does combining retrieval signals improve biomedical evidence recall and answer quality?**